In [1]:
#get data from endpoint
import http.client

conn = http.client.HTTPSConnection("api.collectapi.com")

headers = {
    'content-type': "application/json",
    'authorization': "apikey 5OPjj8xLOygjpPa4Rq7pcr:50NQ3rwuf8WZdwtgUrYPon"
    }

conn.request("GET", "/gasPrice/stateUsaPrice?state=WA", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"success":true,"result":{"state":[{"currency":"usd","name":"Washington","lowername":"washington","gasoline":"5.6060","midGrade":"5.8620","premium":"6.1230","diesel":"6.5600"}],"cities":[{"currency":"usd","gasoline":"5.5120","midGrade":"5.8410","premium":"6.0990","diesel":"6.2240","name":"Bellingham","lowername":"bellingham"},{"currency":"usd","gasoline":"5.7650","midGrade":"6.0040","premium":"6.2220","diesel":"6.5830","name":"Bremerton","lowername":"bremerton"},{"currency":"usd","gasoline":"5.6020","midGrade":"5.7670","premium":"5.9550","diesel":"6.7320","name":"Olympia","lowername":"olympia"},{"currency":"usd","gasoline":"5.2460","midGrade":"5.5250","premium":"5.7590","diesel":"6.2080","name":"Richland-Kennewick-Pasco","lowername":"richland-kennewick-pasco"},{"currency":"usd","gasoline":"5.8570","midGrade":"6.0740","premium":"6.3340","diesel":"6.8140","name":"Seattle-Bellevue-Everett","lowername":"seattle-bellevue-everett"},{"currency":"usd","gasoline":"5.1660","midGrade":"5.4600","p

In [2]:
#install libraries
! uv pip install pandas psycopg2-binary sqlalchemy

Using Python 3.12.3 environment at: gasetl
Checked 3 packages in 10ms


In [3]:
#get the data into python dictionary
import json

json_gasdata = json.loads(data.decode("utf-8"))
cities_data = json_gasdata["result"]["cities"]
cities_data

[{'currency': 'usd',
  'gasoline': '5.5120',
  'midGrade': '5.8410',
  'premium': '6.0990',
  'diesel': '6.2240',
  'name': 'Bellingham',
  'lowername': 'bellingham'},
 {'currency': 'usd',
  'gasoline': '5.7650',
  'midGrade': '6.0040',
  'premium': '6.2220',
  'diesel': '6.5830',
  'name': 'Bremerton',
  'lowername': 'bremerton'},
 {'currency': 'usd',
  'gasoline': '5.6020',
  'midGrade': '5.7670',
  'premium': '5.9550',
  'diesel': '6.7320',
  'name': 'Olympia',
  'lowername': 'olympia'},
 {'currency': 'usd',
  'gasoline': '5.2460',
  'midGrade': '5.5250',
  'premium': '5.7590',
  'diesel': '6.2080',
  'name': 'Richland-Kennewick-Pasco',
  'lowername': 'richland-kennewick-pasco'},
 {'currency': 'usd',
  'gasoline': '5.8570',
  'midGrade': '6.0740',
  'premium': '6.3340',
  'diesel': '6.8140',
  'name': 'Seattle-Bellevue-Everett',
  'lowername': 'seattle-bellevue-everett'},
 {'currency': 'usd',
  'gasoline': '5.1660',
  'midGrade': '5.4600',
  'premium': '5.7100',
  'diesel': '6.2450'

In [4]:
#convert to pandas dataframe
import pandas as pd

citiesdf = pd.DataFrame(cities_data)
citiesdf.columns

Index(['currency', 'gasoline', 'midGrade', 'premium', 'diesel', 'name',
       'lowername'],
      dtype='str')

In [5]:
#drop lowername columns
citiesdf.drop(columns=['lowername'], inplace=True)
citiesdf.columns

Index(['currency', 'gasoline', 'midGrade', 'premium', 'diesel', 'name'], dtype='str')

In [6]:
#rename 'name' column
citiesdf.rename(columns={'name':'city'},inplace=True)
citiesdf.columns

Index(['currency', 'gasoline', 'midGrade', 'premium', 'diesel', 'city'], dtype='str')

In [7]:
#setup database and connection strings
from decouple import config
from sqlalchemy import create_engine,text

#database connection string variables set from .env file
DB_HOST = config('DB_HOST',default='localhost')
DB_PORT = config('DB_PORT',default='5432')
DB_NAME = config('DB_NAME')
DB_USER = config('DB_USER')
DB_PASSWORD = config('DB_PASSWORD')
STAGING_SCHEMA = config('STAGING_SCHEMA')

conn_string = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(conn_string)

In [8]:
#create specific schema
staging_schema = STAGING_SCHEMA
with engine.connect() as conn:
    conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {staging_schema};"))
    conn.commit()

In [9]:
#store to databse
citiesdf.to_sql('cities_gas_prices', engine, if_exists='replace',schema=staging_schema, index=False)


14